In [37]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd


# Define the path to the event log directory
XGBoost_FedXgbBaggin_TR = [
    # "../log/results/XGB/XGBoost_FedXgbBaggin_TR/_2025-03-08_11-48-33/events.out.tfevents.1741463313.ROG-Strix-GL12.1208814.0",
    # "../log/results/XGB/XGBoost_FedXgbBaggin_TR/_2025-03-08_11-49-32/events.out.tfevents.1741463372.ROG-Strix-GL12.1208814.1",
    "../log/results/XGB/XGBoost_FedXgbBaggin_TR/_2025-03-08_11-50-29/events.out.tfevents.1741463429.ROG-Strix-GL12.1208814.2",
    "../log/results/XGB/XGBoost_FedXgbBaggin_TR/_2025-03-08_11-51-25/events.out.tfevents.1741463485.ROG-Strix-GL12.1208814.3",
    "../log/results/XGB/XGBoost_FedXgbBaggin_TR/_2025-03-08_11-52-23/events.out.tfevents.1741463543.ROG-Strix-GL12.1208814.4",
]

XGBoost_FedXgbBaggin_WTR = [
    # "../log/results/XGB/XGBoost_FedXgbBaggin_WTR/_2025-03-08_11-05-43/events.out.tfevents.1741460743.ROG-Strix-GL12.1200954.0",
    # "../log/results/XGB/XGBoost_FedXgbBaggin_WTR/_2025-03-08_11-06-35/events.out.tfevents.1741460795.ROG-Strix-GL12.1200954.1",
    "../log/results/XGB/XGBoost_FedXgbBaggin_WTR/_2025-03-08_11-07-23/events.out.tfevents.1741460843.ROG-Strix-GL12.1200954.2",
    "../log/results/XGB/XGBoost_FedXgbBaggin_WTR/_2025-03-08_11-08-12/events.out.tfevents.1741460892.ROG-Strix-GL12.1200954.3",
    "../log/results/XGB/XGBoost_FedXgbBaggin_WTR/_2025-03-08_11-09-02/events.out.tfevents.1741460942.ROG-Strix-GL12.1200954.4"
]

In [38]:
event = tf.compat.v1.train.summary_iterator(XGBoost_FedXgbBaggin_TR[0])
count = 0
for e in event:
    for v in e.summary.value:
        if v.tag == "Server_Test_F1_Score":
            print(e.step, v.tag, v.simple_value)
            count += 1
            if count == 3:
                break
    if count == 3:
        break

0 Server_Test_F1_Score 0.0
1 Server_Test_F1_Score 0.6419007182121277
2 Server_Test_F1_Score 0.6898440718650818


In [39]:
df_dict = {}
algorithms = { "XGBoost_FedXgbBaggin_TR": XGBoost_FedXgbBaggin_TR, 
              "XGBoost_FedXgbBaggin_WTR": XGBoost_FedXgbBaggin_WTR}

def extract_metrics(algorithm_name, log_files):
    data = {}
    for run_id, log_file in enumerate(log_files):
        if os.path.exists(log_file):
            for event in tf.compat.v1.train.summary_iterator(log_file):
                step = event.step
                key = (step, run_id, algorithm_name)
                if key not in data:
                    data[key] = {
                        "Step": step,
                        "Algorithm": algorithm_name,
                        "Run": run_id  # Track the run index
                    }

                # Extract each metric
                for value in event.summary.value:
                    if value.tag == "Server_Test_F1_Score":
                        data[key]["server_F1_Score"] = value.simple_value
                    elif value.tag == "Server_Test_Loss":
                        data[key]["server_loss"] = value.simple_value
                    elif value.tag == "Server_Test_Precision":
                        data[key]["server_precision"] = value.simple_value
                    elif value.tag == "Server_Test_Recall":
                        data[key]["server_recall"] = value.simple_value
                    elif value.tag == "Server_Test_Accuracy":
                        data[key]["server_accuracy"] = value.simple_value
                    elif value.tag == "Clients_agg/train_accuracy":
                        data[key]["train_accuracy"] = value.simple_value
                    elif value.tag == "Clients_agg/train_loss":
                        data[key]["train_loss"] = value.simple_value
                    elif value.tag == "Clients_agg/train_f1_score":
                        data[key]["train_f1_score"] = value.simple_value
        else:
            raise FileNotFoundError(f"File {log_file} not found")
    # Convert dictionary to a DataFrame
    df = df = pd.DataFrame.from_dict(data, orient="index")

    df = df.fillna(0)  # Fill NaN values with 0

    # Sort by Algorithm, Run, and Step
    df = df.sort_values(by=["Algorithm", "Run", "Step"]).reset_index(drop=True)

    return df if not df.empty else None

# Extract and store DataFrames
df_list = []
for algo_name, files in algorithms.items():
    df = extract_metrics(algo_name, files)
    if df is not None:
        df_list.append(df)

df_all = pd.concat(df_list, ignore_index=True)
df_all.head()

,Step,Algorithm,Run,server_accuracy,server_precision,server_recall,server_F1_Score,server_loss
0,0,XGBoost_FedXgbBaggin_TR,0,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,XGBoost_FedXgbBaggin_TR,0,0.611275,0.836526,0.611275,0.641901,1.871706
2,2,XGBoost_FedXgbBaggin_TR,0,0.680468,0.836333,0.680468,0.689844,1.845208
3,3,XGBoost_FedXgbBaggin_TR,0,0.692810,0.835796,0.692810,0.696410,1.826643
4,4,XGBoost_FedXgbBaggin_TR,0,0.700613,0.834714,0.700613,0.700726,1.811608


In [40]:
# Group by Algorithm and Step, then calculate mean and std for each metric
metrics = ['server_accuracy', 'server_precision', 'server_recall', 'server_F1_Score', 'server_loss']
grouped = df_all.groupby(['Algorithm', 'Step'])[metrics].agg(['mean', 'std']).reset_index()

# Flatten the MultiIndex columns
grouped.columns = ['_'.join(col).strip() if col[1] else col[0] for col in grouped.columns.values]
grouped.head(10)

,Algorithm,Step,server_accuracy_mean,server_accuracy_std,server_precision_mean,server_precision_std,server_recall_mean,server_recall_std,server_F1_Score_mean,server_F1_Score_std,server_loss_mean,server_loss_std
0,XGBoost_FedXgbBaggin_TR,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,XGBoost_FedXgbBaggin_TR,1,0.672506,0.056325,0.856364,0.018659,0.672506,0.056325,0.700133,0.057829,1.846605,0.022107
2,XGBoost_FedXgbBaggin_TR,2,0.718157,0.038996,0.856078,0.019221,0.718157,0.038996,0.731360,0.047230,1.820824,0.021394
3,XGBoost_FedXgbBaggin_TR,3,0.730472,0.040976,0.856641,0.020246,0.730472,0.040976,0.739305,0.048875,1.804080,0.019878
4,XGBoost_FedXgbBaggin_TR,4,0.739125,0.040184,0.857048,0.021278,0.739125,0.040184,0.745397,0.048291,1.790275,0.018766
5,XGBoost_FedXgbBaggin_TR,5,0.743610,0.038495,0.856736,0.021487,0.743610,0.038495,0.747509,0.047563,1.781724,0.017982
6,XGBoost_FedXgbBaggin_TR,6,0.749582,0.036803,0.857301,0.021337,0.749582,0.036803,0.750984,0.046867,1.775109,0.016745
7,XGBoost_FedXgbBaggin_TR,7,0.755766,0.035838,0.857873,0.021809,0.755766,0.035838,0.756184,0.046067,1.769240,0.016173
8,XGBoost_FedXgbBaggin_TR,8,0.756111,0.035696,0.857652,0.020910,0.756111,0.035696,0.755014,0.045949,1.765671,0.016028
9,XGBoost_FedXgbBaggin_TR,9,0.766675,0.039443,0.860047,0.021304,0.766675,0.039443,0.764626,0.049046,1.761064,0.018005


In [41]:
grouped[grouped.Algorithm == "XGBoost_FedXgbBaggin_WTR"].head(10)

,Algorithm,Step,server_accuracy_mean,server_accuracy_std,server_precision_mean,server_precision_std,server_recall_mean,server_recall_std,server_F1_Score_mean,server_F1_Score_std,server_loss_mean,server_loss_std
51,XGBoost_FedXgbBaggin_WTR,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
52,XGBoost_FedXgbBaggin_WTR,1,0.728959,0.011126,0.832288,0.016100,0.728959,0.011126,0.742833,0.014814,1.883734,0.021102
53,XGBoost_FedXgbBaggin_WTR,2,0.728588,0.013338,0.828206,0.019205,0.728588,0.013338,0.742011,0.016820,1.860549,0.022151
54,XGBoost_FedXgbBaggin_WTR,3,0.730791,0.009950,0.825906,0.020958,0.730791,0.009950,0.742125,0.015542,1.843582,0.019978
55,XGBoost_FedXgbBaggin_WTR,4,0.731587,0.007422,0.825262,0.021143,0.731587,0.007422,0.742662,0.013667,1.832911,0.017285
56,XGBoost_FedXgbBaggin_WTR,5,0.736258,0.009378,0.825964,0.021413,0.736258,0.009378,0.745637,0.014016,1.824326,0.016525
57,XGBoost_FedXgbBaggin_WTR,6,0.736577,0.009895,0.825536,0.022314,0.736577,0.009895,0.745553,0.014457,1.819274,0.015571
58,XGBoost_FedXgbBaggin_WTR,7,0.738116,0.008907,0.823895,0.024350,0.738116,0.008907,0.745586,0.014117,1.816357,0.014659
59,XGBoost_FedXgbBaggin_WTR,8,0.738567,0.008443,0.823235,0.023646,0.738567,0.008443,0.745775,0.013936,1.814556,0.013899
60,XGBoost_FedXgbBaggin_WTR,9,0.739364,0.009359,0.823023,0.024672,0.739364,0.009359,0.746182,0.015014,1.813880,0.013701


In [42]:
df_mean_std = grouped.copy()

In [43]:
# List of metrics to analyze (excluding 'Step' and 'Algorithm')
metrics = [col for col in df_mean_std.columns if col not in ["Algorithm", "Step"] and not "std" in col and not "loss" in col]

# Initialize a dictionary to store results
max_results = []

# Loop through each algorithm and each metric
for algo in df_mean_std["Algorithm"].unique():
    df_algo = df_mean_std[df_mean_std["Algorithm"] == algo]
    
    for metric in metrics:
        max_row = df_algo.loc[df_algo[metric].idxmax()]  # Get the row where max happens
        max_results.append({
            "Algorithm": algo,
            "Metric": metric,
            "Max Value": max_row[metric],
            "Step": max_row["Step"]
        })

# Convert results into a DataFrame
df_max_metrics = pd.DataFrame(max_results)

df_max_metrics

,Algorithm,Metric,Max Value,Step
0,XGBoost_FedXgbBaggin_TR,server_accuracy_mean,0.798896,34
1,XGBoost_FedXgbBaggin_TR,server_precision_mean,0.866057,34
2,XGBoost_FedXgbBaggin_TR,server_recall_mean,0.798896,34
3,XGBoost_FedXgbBaggin_TR,server_F1_Score_mean,0.788768,34
4,XGBoost_FedXgbBaggin_WTR,server_accuracy_mean,0.739364,9
5,XGBoost_FedXgbBaggin_WTR,server_precision_mean,0.833583,16
6,XGBoost_FedXgbBaggin_WTR,server_recall_mean,0.739364,9
7,XGBoost_FedXgbBaggin_WTR,server_F1_Score_mean,0.746182,9


In [46]:
# algorithm = "XGBoost_FedXgbBaggin_TR"
algorithm = "XGBoost_FedXgbBaggin_WTR"
step = 9
row = df_mean_std[(df_mean_std["Algorithm"] == algorithm) & (df_mean_std["Step"] == step)]
metrics = ["server_accuracy_mean", "server_precision_mean", "server_recall_mean", "server_F1_Score_mean", ]

# metrics_values = {metric: round(row[metric].values[0] * 100, 2) for metric in metrics}
metrics_values = {metric: round(row[metric].values[0] , 4) for metric in metrics}
metrics_values

{'server_accuracy_mean': 0.7394,
 'server_precision_mean': 0.823,
 'server_recall_mean': 0.7394,
 'server_F1_Score_mean': 0.7462}

In [ ]:
{'server_accuracy_mean': 0.7394,
 'server_precision_mean': 0.823,
 'server_recall_mean': 0.7394,
 'server_F1_Score_mean': 0.7462}

{'server_accuracy_mean': 0.7976,
 'server_precision_mean': 0.8598,
 'server_recall_mean': 0.7976,
 'server_F1_Score_mean': 0.7912}


